# 使用 Microsoft Foundry 微調模型

此筆記本遵循當前的 [Microsoft Foundry 微調工作流程](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning?WT.mc_id=academic-105485-koreyst)。它使用指向您的 Foundry 資源的 `/openai/v1/` 端點的 **OpenAI Python SDK (v1)**，因此相同的代碼只需更改客戶端設置即可在 OpenAI 平台上運行。

> <strong>先決條件</strong>
> - 擁有一個 [Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) 專案，並具有 **Foundry 擁有者** 角色（微調和部署所需）。
> - `pip install "openai>=1.0" python-dotenv`
> - 一個包含 `AZURE_OPENAI_ENDPOINT` 和 `AZURE_OPENAI_API_KEY` 的 `.env` 文件（參見 [課程設置指南](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)）。
> - 一個目前支援的基礎模型，如 `gpt-4.1-mini`（參見 [可微調模型列表](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models)）。

微調通過使用與您的任務相關的其他範例重新訓練基礎模型來提升模型。提示工程技術如 _少量示例學習_ 和 _檢索增強生成_ 可用相關資料豐富提示，但它們受模型上下文窗口的限制。

使用微調，您直接重新訓練模型本身（使用遠多於上下文窗口能容納的示例），並部署一個 _自訂_ 版本，推論時不再需要那些示例。這提高了質量，釋放上下文窗口，並通過縮短提示可降低成本和延遲。在底層，Foundry 使用 **LoRA（低秩調整）** 進行高效訓練。

Foundry 支援三種技術：本教程使用的 **監督微調（SFT）**，以及 **DPO**（偏好對齊）和 **RFT**（強化微調）。工作流程分為四步：

1. 準備並上傳您的訓練 <strong>和驗證</strong> 資料。
2. 運行訓練任務以建立微調模型。
3. 監控任務，檢視指標，並選擇檢查點。
4. 部署微調模型並用於推論。

本教程中，我們使用 SFT 微調 `gpt-4.1-mini`，建立一個用 limerick 回答元素週期表問題的聊天機器人。

---


### 第 1.1 步：準備您的數據集

讓我們構建一個聊天機器人，通過用五行打油詩回答有關元素的問題，幫助您理解元素周期表。在_本_簡單教程中，我們將僅創建一個數據集，用幾個示例回應來訓練模型，展示數據的預期格式。在實際應用中，您需要創建一個包含更多示例的數據集。如果存在開放數據集（符合您的應用領域），您也可以使用並重新格式化它以用於微調。

由於我們專注於`gpt-4.1-mini`並尋求單輪回應（聊天完成），我們可以使用[這個建議格式](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst)來創建示例，該格式符合 OpenAI 聊天完成的要求。如果您期待多輪對話內容，您將使用[多輪示例格式](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst)，其中包括一個`weight`參數，以標示在微調過程中應該使用（或不使用）的訊息。

我們將為本教程使用較簡單的單輪格式。數據使用[jsonl 格式](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst)，每行包含一條記錄，每條記錄表示為 JSON 格式的物件。下面的代碼片段顯示了兩條記錄作為示例 - 查看[training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl)以獲得我們微調教程使用的完整示例集（共 10 個範例）。<strong>注意：</strong>每條記錄_必須_定義在單行內（不像格式化 JSON 文件那樣跨多行）。

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

在實際應用中，您將需要更大規模的示例集才能取得良好效果—這會在回應質量與微調所需時間/成本之間做權衡。我們使用較小的示例集，以便可以快速完成微調，展示整個過程。請參考[這個 OpenAI Cookbook 範例](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst)以了解更複雜的微調教學。


---

### 步驟 1.2：上傳你的數據集

使用 Files API (`purpose="fine-tune"`) 將你的訓練和驗證檔案上傳至 Foundry。提供<strong>驗證檔案</strong>可讓 Foundry 報告驗證損失，方便你發現過擬合情況。

在執行以下程式碼前，請確保你已經：
 - 安裝 SDK：`pip install "openai>=1.0" python-dotenv`
 - 建立包含 `AZURE_OPENAI_ENDPOINT` 和 `AZURE_OPENAI_API_KEY` 的 `.env` 檔案

程式碼會建立一個指向你 Foundry 資源 `/openai/v1/` 端點的 OpenAI 用戶端，然後上傳兩個檔案並列印其 ID。


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Point the OpenAI SDK at your Microsoft Foundry resource's /openai/v1/ endpoint.
# (For the OpenAI platform instead, use: client = OpenAI()  with OPENAI_API_KEY set.)
client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
)

training_response = client.files.create(
    file=open("./training-data.jsonl", "rb"), purpose="fine-tune"
)
validation_response = client.files.create(
    file=open("./validation-data.jsonl", "rb"), purpose="fine-tune"
)

training_file_id = training_response.id
validation_file_id = validation_response.id

print("Training file ID:", training_file_id)
print("Validation file ID:", validation_file_id)


---

### 步驟 2.1：使用 SDK 建立微調工作


In [ ]:
# Create the fine-tuning job.
# trainingType "GlobalStandard" is the recommended tier (lower cost, faster queue).
# Use "Standard" if you need regional data residency, or "Developer" for cheap experiments.
job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini",              # base model to fine-tune
    training_file=training_file_id,
    validation_file=validation_file_id,
    suffix="elements-limerick",        # helps you identify the resulting model
    seed=105,                          # makes the run reproducible
    extra_body={"trainingType": "GlobalStandard"},
)

job_id = job.id
print("Fine-tuning Job ID:", job_id)
print("Status:", job.status)


---

### 步驟 2.2：檢查工作的狀態

這裡有一些你可以使用 `client.fine_tuning.jobs` API 做的事情：
- `client.fine_tuning.jobs.list(limit=<n>)` - 列出最近的 n 個微調工作
- `client.fine_tuning.jobs.retrieve(<job_id>)` - 獲取特定工作的詳細資料
- `client.fine_tuning.jobs.cancel(<job_id>)` - 取消一個工作
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<n>)` - 列出該工作中的事件
- `client.fine_tuning.jobs.checkpoints.list(<job_id>)` - 列出可部署的檢查點（最近幾個回合）

當工作開始時，Foundry 首先會_驗證訓練文件_以確保資料格式正確。然後進行訓練，耗時可能從幾分鐘到幾小時不等，視模型和資料集大小而定。


In [ ]:
# List the last 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the current state of your fine-tuning job
client.fine_tuning.jobs.retrieve(job_id)

# List up to 10 events from the job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)


In [ ]:
# Track the job status until it is complete
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)


---

### 步驟 2.3：追蹤事件以監察進度


In [ ]:
# Track progress in a more granular way by checking events.
# Re-run this cell until you see "The job has successfully completed".
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)


In [ ]:
# When the job finishes, the last few epochs are available as deployable checkpoints.
# Best practice: don't blindly deploy the final epoch - review the checkpoints and pick
# the one that generalizes best (watch train_loss vs. valid_loss and token accuracy).
checkpoints = client.fine_tuning.jobs.checkpoints.list(job_id)
for cp in checkpoints.data:
    print(cp.fine_tuned_model_checkpoint, "| step:", cp.step_number)


### 第 2.4 步：在 Foundry 門戶中檢閱狀態、指標和檢查點


您亦可於 [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) 的 **建置 > 微調** 中跟踪該工作。選擇您的工作以查看其狀態、訓練事件、超參數和指標。請留意這些指標：

- `train_loss` 與 `full_valid_loss` - 應隨時間減少。
- `train_mean_token_accuracy` 與 `full_valid_mean_token_accuracy` - 應增加。

若訓練曲線與驗證曲線背離，可能為過擬合 – 可嘗試較少的訓練週期或較低的學習率乘數。下圖示範您將看到的狀態、訊息和指標面板類型（具體界面依服務提供者而異）。

![微調工作狀態](../../../../../translated_images/zh-MO/fine-tuned-model-status.563271727bf7bfba.webp)


你亦可以向下捲動視覺化儀表板以檢視狀態訊息及指標，如以下所示：

| Messages | Metrics |
|:---|:---|
| ![Messages](../../../../../translated_images/zh-MO/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/zh-MO/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### 第 3.1 步：取得微調模型 ID

當作業成功時，`response.fine_tuned_model` 會儲存您自訂模型的 ID（例如，`gpt-4.1-mini-2025-04-14.ft-...`）。在 Azure 上，您必須先 <strong>部署</strong> 該模型，才能調用它。


In [ ]:
# Retrieve the id of the fine-tuned model once the job has succeeded
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)


### 步驟 3.2：部署微調模型

不同於 OpenAI 平台（您可以直接呼叫微調模型 ID），Microsoft Foundry 需要您先<strong>部署</strong>模型。最簡單的方法是使用[Foundry 入口網站](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)：進入 **建置 > 微調**，選擇您已完成的工作，然後選擇 <strong>部署</strong>。為部署命名（例如 `elements-limerick`）— 該部署名稱即是您從程式碼中呼叫的名稱。

您也可以透過控制平面 REST/ARM API 程式化部署；詳情請參閱[部署指南](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning-deploy?WT.mc_id=academic-105485-koreyst)。請記得已部署的自訂模型會按小時計費，且非活躍部署會於 15 天後被移除。


In [ ]:
# Once deployed, call your fine-tuned model by its DEPLOYMENT name via the Responses API.
# (On the OpenAI platform you'd pass fine_tuned_model_id directly instead.)
deployment_name = "elements-limerick"  # the name you gave the deployment in Foundry

completion = client.responses.create(
    model=deployment_name,
    input=[
        {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
        {"role": "user", "content": "Tell me about Strontium"},
    ],
    store=False,
)
print(completion.output_text)


---

### 步驟 3.3：在 Foundry 遊樂場測試你的微調模型

你亦可以在 [Microsoft Foundry 入口網站](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)的<strong>遊樂場</strong>內測試已部署的模型—從模型下拉選單選擇你的微調部署，並嘗試輸入提示。請使用你訓練時所用的<strong>同一個系統訊息</strong>；不同的系統訊息會改變行為。

> **提示：** 將微調模型與基礎版 `gpt-4.1-mini` 並排比較。留意微調模型如何以你範例中的打油詩格式回應答案，而基礎模型僅遵循系統提示。

**這是一個刻意簡單的範例來展示流程，並非真實世界數據集。** 在生產環境中，你會針對真實資料（例如，用於客戶服務的產品目錄）進行微調，質量差異會更加明顯—且單靠提示工程來匹配該質量，呼叫時的代幣消耗會高得多。欲獲更完整範例，請參閱 [OpenAI Cookbook 微調指南](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) 及 [Foundry 微調教學](https://learn.microsoft.com/azure/ai-foundry/openai/tutorials/fine-tune?WT.mc_id=academic-105485-koreyst)。

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
